In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import xarray as xr

from fast_o3_iv.modeling.train import DatasetConfig
import fast_o3_iv.bias_correction as bc
import fast_o3_iv.attributable_fraction as af

In [2]:
config_hist = DatasetConfig(
            source_id = 'CESM2-WACCM6',
            cmip_experiment_id = 'historical',
            variable_setname = 'setC',
            year_range = range(1975,2015)
        )
config_ssp245 = DatasetConfig(
            source_id = 'CESM2-WACCM6',
            cmip_experiment_id = 'ssp245',
            variable_setname = 'setC',
            year_range = range(2015,2055)
        )

In [3]:
file_path = bc.return_osdma8_bc_path("inference", config_hist, "GPR-local")
wa6_hist_osdma8_bc = xr.open_dataset(file_path)

file_path = bc.return_osdma8_bc_path("inference", config_ssp245, "GPR-local")
wa6_ssp245_osdma8_bc = xr.open_dataset(file_path)

In [10]:
wa6_hist_AF = af.calc_af(wa6_hist_osdma8_bc)
wa6_ssp245_AF = af.calc_af(wa6_ssp245_osdma8_bc)

In [11]:
wa6_ssp245_AF

<xarray.Dataset> Size: 531MB
Dimensions:          (ensemble_member: 10, lat: 144, lon: 288, time: 40)
Coordinates:
  * ensemble_member  (ensemble_member) <U3 120B '001' '002' ... '009' '010'
  * lat              (lat) float64 1kB -59.84 -58.9 -57.96 ... 73.04 73.98 74.92
  * lon              (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
  * time             (time) datetime64[ns] 320B 2015-01-01 ... 2054-01-01
Data variables:
    y_true_bc_af     (time, lat, lon, ensemble_member) float64 133MB 0.0 ... ...
    y_true_af        (time, lat, lon, ensemble_member) float64 133MB 0.0 ... ...
    y_pred_bc_af     (time, lat, lon, ensemble_member) float64 133MB 0.0 ... ...
    y_pred_af        (time, lat, lon, ensemble_member) float64 133MB 0.0 ... ...
Attributes:
    af:                            attributable fraction, the percentage of d...
    units:                         percent
    TMREL:                         theoretical minimum risk exposure level, 3...
    RR:                            1.063 for each 10 ppbv increase in ozone a...
    beta:                          log-linear RR slope = ln(1.063) / 10 ppbv
    health-relevant ozone metric:  OSDMA8
    methodology:                   GBD 2019 methodology (Murray et al., 2020)

In [13]:
file_path = af.return_af_path("inference", config_hist, "GPR-local")
wa6_hist_AF.to_netcdf(file_path)

file_path = af.return_af_path("inference", config_ssp245, "GPR-local")
wa6_ssp245_AF.to_netcdf(file_path)